# Agrimix Northern Soils — example queries

This notebook shows common analytical patterns over the DuckDB database. Run cells top-to-bottom.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

DB = Path('..') / 'db' / 'agrimix.duckdb'
con = duckdb.connect(str(DB), read_only=True)
con.sql("INSTALL spatial; LOAD spatial;")
print(con.sql("SHOW TABLES").df())

## Region overview

In [ ]:
con.sql("SELECT * FROM v_region_summary ORDER BY cent_lat DESC").df()

## Vertosol distribution by rainfall band, ranked by region

In [ ]:
con.sql("""
SELECT nrm_region, band_label, ROUND(hectares) AS ha
FROM soil_rain_coarse
WHERE soil_name = 'Vertosol'
  AND hectares > 100000
ORDER BY hectares DESC
""").df()

## Cross-tab for one region (50 mm bands), wide format

In [ ]:
df = con.sql("""
SELECT soil_name, band_label, hectares
FROM soil_rain_fine
WHERE nrm_region = 'Fitzroy' AND hectares > 0
""").df()
pivot = df.pivot(index='soil_name', columns='band_label', values='hectares').fillna(0).astype(int)
pivot

## Carbon-friendly country: cracking clays + reliable rainfall

Vertosols, Dermosols and Ferrosols in regions getting >500 mm — typical SOC sequestration potential.

In [ ]:
con.sql("""
SELECT nrm_region,
       SUM(CASE WHEN soil_name='Vertosol' THEN hectares END) AS vertosol_ha,
       SUM(CASE WHEN soil_name='Dermosol' THEN hectares END) AS dermosol_ha,
       SUM(CASE WHEN soil_name='Ferrosol' THEN hectares END) AS ferrosol_ha,
       SUM(hectares) AS total_above_500mm_ha
FROM soil_rain_coarse
WHERE band_low_mm >= 500
  AND soil_name IN ('Vertosol','Dermosol','Ferrosol')
GROUP BY nrm_region
ORDER BY total_above_500mm_ha DESC
""").df()

## Total estimated land asset value per region

In [ ]:
con.sql("SELECT * FROM v_total_estimated_value").df()